# SDAR Attack on FedProto

Make sure your runtime has a GPU enabled (**Runtime > Change runtime type > T4 GPU**).

In [ ]:
import os

# Clone if it doesn't exist, otherwise pull latest changes
if not os.path.exists('prototype_based_fl'):
    !git clone https://github.com/ekinkyl/prototype_based_fl.git
    %cd prototype_based_fl/fedproto_sdar
else:
    %cd prototype_based_fl/fedproto_sdar
    !git pull

# Install the required packages
!pip install -r requirements.txt

---
### Run the SDAR Attack (White-Box Simulator)
This runs the attack using ResNet18 as the server's Simulator (exact copy of the client model).

At the end of training, it will automatically compute and print:
- **MSE** (lower is better)
- **PSNR** in dB (higher is better)
- **SSIM** (higher is better, max 1.0)
- **Downstream Classifier Accuracy** (higher = more info leaked)
- Per-class breakdown for each evaluated client

In [ ]:
!python scripts/run_fedproto_sdar.py \
    --dataset cifar10 \
    --model resnet18 \
    --num_users 20 \
    --rounds 50 \
    --ways 5 \
    --shots 100 \
    --ld 0.1 \
    --local_bs 32 \
    --attack \
    --attack_epochs 5 \
    --lambda1 0.02 \
    --lambda2 1e-5

---
### View Reconstructions
Displays all saved reconstruction images in chronological order (Round 0 → Round 49).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

images = sorted(glob.glob('results/reconstructions/*.png'))

if images:
    for img_path in images:
        print(f"\n{os.path.basename(img_path)}")
        img = mpimg.imread(img_path)
        plt.figure(figsize=(15, 3))
        plt.imshow(img)
        plt.axis('off')
        plt.show()
else:
    print("No reconstructions found yet. Run the training first!")

---
### Load and Display Saved Attack Metrics
If you want to reload the metrics from a previous run without retraining.

In [ ]:
import numpy as np

metrics_path = 'results/attack_metrics.npy'
if os.path.exists(metrics_path):
    metrics = np.load(metrics_path, allow_pickle=True).item()
    print('=== Attack Metrics ===')
    print(f"MSE  : {np.mean(metrics['mse']):.6f} ± {np.std(metrics['mse']):.6f}")
    print(f"PSNR : {np.mean(metrics['psnr']):.2f} ± {np.std(metrics['psnr']):.2f} dB")
    print(f"SSIM : {np.mean(metrics['ssim']):.4f} ± {np.std(metrics['ssim']):.4f}")
    if metrics.get('downstream_acc'):
        print(f"Downstream Acc: {np.mean(metrics['downstream_acc']):.4f}")
    print('\n=== Per-Client, Per-Class MSE ===')
    for client_key, class_mses in metrics.get('per_class_mse', {}).items():
        print(f"  Client {client_key}:")
        for cls, mse_val in sorted(class_mses.items()):
            print(f"    Class {cls}: MSE = {mse_val:.6f}")
else:
    print('No saved metrics found. Run the training first!')

---
### Alternative Experiments

In [ ]:
# Run FedProto + SDAR Attack (Black-Box Surrogate Simulator)
!python scripts/run_fedproto_sdar.py \
    --dataset cifar10 \
    --model resnet18 \
    --num_users 20 \
    --rounds 50 \
    --ways 5 \
    --shots 100 \
    --ld 0.1 \
    --local_bs 32 \
    --attack \
    --diff_simulator \
    --attack_epochs 5 \
    --lambda1 0.02 \
    --lambda2 1e-5

In [ ]:
# Run FedProto baseline (No Attack)
!python scripts/run_fedproto_baseline.py \
    --dataset cifar10 \
    --model resnet18 \
    --num_users 20 \
    --rounds 50 \
    --ways 5 \
    --shots 100 \
    --ld 0.1 \
    --local_bs 32